# PI-Link QML, round 2: transformer & LLM baselines + tuned 12-qubit hardware run
**Runtime → Change runtime type → T4 GPU.** Run the cells in order. Results save as they go, so if Colab disconnects, re-run from the top: finished parts are skipped.

Total: about 1.5–2 hours for Part A (keep the tab open), and a few minutes of QPU time for Part B.

Attach `round2_results.zip` back in the chat at the end.

In [ ]:
from google.colab import files
up = files.upload()  # choose colab_bundle2.zip
!unzip -oq colab_bundle2.zip && ls

In [ ]:
!pip -q install qiskit==2.* qiskit-aer qiskit-ibm-runtime xgboost lightgbm
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

## Part A: transformer and LLM baselines (same splits as the paper)
First a 2-minute smoke test, so any problem shows up before the long run.

In [ ]:
!python colab_eval.py --data pilink_pairs_meta.csv --split repo --out results_tf/pilink --smoke

PI-Link: 7 seeds (cross-encoders on 800 pairs), plus 3 seeds of a cross-encoder on all training pairs and 3 seeds of the LLM reranker. About 1 hour.

In [ ]:
!python -u colab_eval.py --data pilink_pairs_meta.csv --split repo --out results_tf/pilink 2>&1 | grep -v Warning | tee tf_pilink.log | grep -E 'seed|AUC|Error|error' 

In [ ]:
!zip -qr round2_results.zip results_tf tf_pilink.log
files.download('round2_results.zip')  # safety copy

GHPR (CNCF projects, time-based split): same models. About 45 minutes.

In [ ]:
!python -u colab_eval.py --data ghpr_pairs_meta.csv --split temporal --out results_tf/ghpr --max_full 20000 2>&1 | grep -v Warning | tee tf_ghpr.log | grep -E 'seed|AUC|Error|error' 

In [ ]:
!zip -qr round2_results.zip results_tf tf_pilink.log tf_ghpr.log
files.download('round2_results.zip')

## Part B: real IBM QPU, final tuned model (QSVC-Z, 12 qubits)
852 circuits × 1,024 shots, about 3–4 QPU-minutes. The open plan gives 10 free QPU-minutes per month. If the job is refused for lack of time, it can wait until next month: Part A results are already saved.

Paste your **new** API key in the hidden prompt (not the one shared in chat).

In [ ]:
import getpass
TOKEN = getpass.getpass('IBM Quantum API key: ')
INSTANCE = 'PASTE_YOUR_INSTANCE_CRN_HERE'

In [ ]:
!python -u hw_final.py --token {TOKEN} --instance {INSTANCE} --n 24 --shots 1024 --out hw_final 2>&1 | grep -v Warning
!zip -qr round2_results.zip results_tf tf_pilink.log tf_ghpr.log hw_final
files.download('round2_results.zip')